<a href="https://colab.research.google.com/github/guuhh/AOC-2019.2/blob/master/Spark_estudos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark import SparkContext
spark_context = SparkContext()
print(spark_context)
print(spark_context.version)
spark_context.stop()

<SparkContext master=local[*] appName=pyspark-shell>
4.0.4


In [ ]:
from pyspark.sql import SparkSession
#spark_session = SparkSession.builder.getOrCreate()
spark_session = SparkSession.builder.master("local").appName("Word Count").getOrCreate()
print(spark_session)
dataset = spark_session.read.csv('/content/sample_data/california_housing_train.csv',
                                 inferSchema=True, header=True)
#dataset.head()
#dataset.count()
#dataset.printSchema()
#dataset.show()
#spark_session.stop()

# create a temporary table and list it
dataset.createOrReplaceTempView('temporary_table')
print(spark_session.catalog.listTables())
#O próximo passo consiste em fazer uma consulta SQL.
#Para isso, vamos selecionar apenas três registros com os dados referentes às colunas “longitude” e “latitude”, executando o seguinte código:
query = 'FROM temporary_table SELECT longitude, latitude LIMIT 3'
#instacie outputquery
output_query = spark_session.sql(query)
output_query.show()
#spark_session.stop()

[Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
+---------+--------+
|longitude|latitude|
+---------+--------+
|  -114.31|   34.19|
|  -114.47|    34.4|
|  -114.56|   33.69|
+---------+--------+



In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

spark_session = SparkSession.builder.master("local").appName("Max_qty_rooms").getOrCreate()
dataset = spark_session.read.csv(
    '/content/sample_data/california_housing_train.csv',
    inferSchema=True,
    header=True)
dataset.createOrReplaceTempView("temporary_table")
print(spark_session.catalog.listTables())

query = 'SELECT MAX(total_rooms) as maximo_quartos FROM temporary_table'
q_maximo_quartos = spark_session.sql(query)
q_maximo_quartos.show()
maximo = q_maximo_quartos.collect()[0]['maximo_quartos']
print('Resultado do spark SQL:',maximo)
pd_maximo_quartos = q_maximo_quartos.toPandas()
print('Quantidade maxima de quartos no Pandas é :{}'.format(pd_maximo_quartos['maximo_quartos']))
print('Quantidade maxima de quartos no Pandas é - apenas imprimir primeira linha :{}'.format(pd_maximo_quartos['maximo_quartos'].iloc[0]))
print('Quantidade maxima de quartos no Pandas é - apenas imprimir primeira e segunda linha :{}'.format(pd_maximo_quartos['maximo_quartos'].iloc[0:2]))
qty_maximo_quartos = int(pd_maximo_quartos.loc[0,'maximo_quartos'])
print('Resultado do Pandas:',qty_maximo_quartos)

query2 = 'SELECT longitude, latitude FROM temporary_table WHERE total_rooms = '+str(qty_maximo_quartos)
localizacao_maximo_quartos = spark_session.sql(query2)
pd_localizacao_maximo_quatos = localizacao_maximo_quartos.toPandas()
print(pd_localizacao_maximo_quatos.head())

media = 0
desvio_padrao = 0.1
pd_temporario = pd.DataFrame(np.random.normal(media, desvio_padrao, 100))
spark_temporario =  spark_session.createDataFrame(pd_temporario)

print(spark_session.catalog.listTables())
spark_temporario.createOrReplaceTempView('new_temporary_table')
print(spark_session.catalog.listTables())
spark_session.stop()



[Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
+--------------+
|maximo_quartos|
+--------------+
|       37937.0|
+--------------+

Resultado do spark SQL: 37937.0
Quantidade maxima de quartos no Pandas é :0    37937.0
Name: maximo_quartos, dtype: float64
Quantidade maxima de quartos no Pandas é - apenas imprimir primeira linha :37937.0
Quantidade maxima de quartos no Pandas é - apenas imprimir primeira e segunda linha :0    37937.0
Name: maximo_quartos, dtype: float64
Resultado do Pandas: 37937
   longitude  latitude
0    -117.74     33.89
[Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
[Table(name='new_temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


In [25]:
from pyspark import SparkContext
import numpy as np
from operator import add

# Usamos getOrCreate() para recuperar o contexto existente ou criar um novo se não houver
spark_context = SparkContext.getOrCreate()

#vetor = np.array([10, 20, 30, 40, 50])
# Usando lista pura do Python []
vetor = [10, 20, 30, 40, 50]
paralelo = spark_context.parallelize(vetor)

print(paralelo)
mapa = paralelo.map(lambda x : x**2+x)
values = mapa.collect()
print(values)
sum = mapa.reduce(add)
print(sum)
spark_context.stop()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:298
[110, 420, 930, 1640, 2550]
5650


In [22]:
from pyspark import SparkContext
import numpy as np

# Usamos getOrCreate() para recuperar o contexto existente ou criar um novo se não houver
spark_context = SparkContext.getOrCreate()

paralelo = spark_context.parallelize(['distribuida', 'distribuida', 'spark', 'rdd', 'spark','spark'])

funcao_lambda = lambda x:(x,1)

#Na primeira linha, importamos o operador “add” que será usado para somar as ocorrências das palavras mapeadas.
#Já na segunda linha, muitas coisas estão ocorrendo:
#Fazemos o mapeamento dos dados da variável “paralelo” para a função lambda, ou seja, para cada palavra criamos um par (palavra, 1).
#Em seguida, aplicamos a redução com a função “reduceKey”, que soma as ocorrências e as agrupa pela chave, no caso, pelas palavras.
#Na etapa final, dada pela função “collect”, fazemos a coleta dos dados em uma lista que chamamos de “mapa”.
#
from operator import add
mapa = paralelo.map(funcao_lambda).reduceByKey(add).collect()
for (w, c) in mapa:
     print('{}: {}'.format(w, c))
spark_context.stop()

rdd: 1
distribuida: 2
spark: 3


In [29]:
#Agora, vamos desenvolver um exemplo que envolve as operações de transformações e de ações.
#O primeiro passo é instanciar um SparkContext, utilizando o código abaixo:

from pyspark import SparkContext
spark_context = SparkContext.getOrCreate()

list = [1,2,3,4,5,3]
list_rdd = spark_context.parallelize(list)

total_qty = list_rdd.count()
print(total_qty)

#Agora, vamos criar uma função lambda que recebe um número como parâmetro
#e retorna um par formador pelo número do parâmetro e pelo mesmo número multiplicado por 10

par_ordered = lambda number: (number, number*10)

#Nesse próximo passo, vamos aplicar a transformação “flatMap”
#com a ação “collect” da função lambda para a “list_rdd”:

list_rdd_ordered = list_rdd.flatMap(par_ordered).collect()
print(list_rdd_ordered)

#Agora, vamos aplicar a transformação “map”
#com a ação “collect” da função lambda para a “list_rdd”:

list_rdd_map = list_rdd.map(par_ordered).collect()
print(list_rdd_map)

spark_context.stop()

6
[1, 10, 2, 20, 3, 30, 4, 40, 5, 50, 3, 30]
[(1, 10), (2, 20), (3, 30), (4, 40), (5, 50), (3, 30)]
